In [1]:
import requests
import pandas as pd
import json
from datetime import datetime

print("Library siap!")

Library siap!


In [2]:
# Koordinat titik-titik pemantauan kawasan Danau Toba
LOCATIONS = {
    "balige"     : {"lat": -2.3333, "lon": 99.0667, "label": "Balige (Tobasa)"},
    "parapat"    : {"lat": -2.6600, "lon": 98.9400, "label": "Parapat"},
    "pangururan" : {"lat": -2.5900, "lon": 98.6900, "label": "Pangururan (Samosir)"},
    "nainggolan" : {"lat": -2.6300, "lon": 98.8100, "label": "Nainggolan"},
    "tengah_danau": {"lat": -2.6000, "lon": 98.8000, "label": "Tengah Danau Toba"},
}

print("Lokasi pemantauan:")
for key, val in LOCATIONS.items():
    print(f"  {val['label']:30s} → lat: {val['lat']}, lon: {val['lon']}")

Lokasi pemantauan:
  Balige (Tobasa)                → lat: -2.3333, lon: 99.0667
  Parapat                        → lat: -2.66, lon: 98.94
  Pangururan (Samosir)           → lat: -2.59, lon: 98.69
  Nainggolan                     → lat: -2.63, lon: 98.81
  Tengah Danau Toba              → lat: -2.6, lon: 98.8


In [3]:
def fetch_historical_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude"                   : lat,
        "longitude"                  : lon,
        "start_date"                 : start_date,
        "end_date"                   : end_date,
        "daily"                      : [
            "temperature_2m_max",        # Suhu maksimum harian (°C)
            "temperature_2m_min",        # Suhu minimum harian (°C)
            "temperature_2m_mean",       # Suhu rata-rata harian (°C)
            "precipitation_sum",         # Curah hujan harian (mm)
            "windspeed_10m_max",         # Kecepatan angin maksimum (km/h)
            "winddirection_10m_dominant",# Arah angin dominan (°)
            "weathercode",               # Kode kondisi cuaca
            "et0_fao_evapotranspiration",# Evapotranspirasi (untuk pertanian)
        ],
        "timezone": "Asia/Jakarta"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

print("Fungsi fetch_historical_weather siap!")

Fungsi fetch_historical_weather siap!


In [4]:
START_DATE = "2015-01-01"
END_DATE   = "2024-12-31"

all_data = {}

for location_key, location_info in LOCATIONS.items():
    print(f"Mengambil data: {location_info['label']}...", end=" ")

    raw = fetch_historical_weather(
        lat        = location_info["lat"],
        lon        = location_info["lon"],
        start_date = START_DATE,
        end_date   = END_DATE
    )

    if raw:
        df = pd.DataFrame(raw["daily"])
        df["date"]     = pd.to_datetime(df["time"])
        df["location"] = location_info["label"]
        df["lat"]      = location_info["lat"]
        df["lon"]      = location_info["lon"]
        df = df.drop(columns=["time"])

        all_data[location_key] = df
        print(f"✓ {len(df)} baris data ({df['date'].min().date()} s/d {df['date'].max().date()})")
    else:
        print("✗ Gagal")

print(f"\nTotal lokasi berhasil diambil: {len(all_data)}/{len(LOCATIONS)}")

Mengambil data: Balige (Tobasa)... ✓ 3653 baris data (2015-01-01 s/d 2024-12-31)
Mengambil data: Parapat... ✓ 3653 baris data (2015-01-01 s/d 2024-12-31)
Mengambil data: Pangururan (Samosir)... ✓ 3653 baris data (2015-01-01 s/d 2024-12-31)
Mengambil data: Nainggolan... ✓ 3653 baris data (2015-01-01 s/d 2024-12-31)
Mengambil data: Tengah Danau Toba... ✓ 3653 baris data (2015-01-01 s/d 2024-12-31)

Total lokasi berhasil diambil: 5/5


In [5]:
# Gabungkan semua dataframe
df_all = pd.concat(all_data.values(), ignore_index=True)

# Urutkan berdasarkan lokasi dan tanggal
df_all = df_all.sort_values(["location", "date"]).reset_index(drop=True)

# Simpan ke CSV
output_path = "../data/raw/weather_toba_2015_2024.csv"
df_all.to_csv(output_path, index=False)

print(f"Data berhasil disimpan ke: {output_path}")
print(f"\nRingkasan dataset:")
print(f"  Total baris  : {len(df_all):,}")
print(f"  Total kolom  : {len(df_all.columns)}")
print(f"  Periode      : {df_all['date'].min().date()} s/d {df_all['date'].max().date()}")
print(f"  Lokasi       : {df_all['location'].nunique()} titik")
print(f"\nKolom tersedia:")
for col in df_all.columns:
    print(f"  - {col}")

Data berhasil disimpan ke: ../data/raw/weather_toba_2015_2024.csv

Ringkasan dataset:
  Total baris  : 18,265
  Total kolom  : 12
  Periode      : 2015-01-01 s/d 2024-12-31
  Lokasi       : 5 titik

Kolom tersedia:
  - temperature_2m_max
  - temperature_2m_min
  - temperature_2m_mean
  - precipitation_sum
  - windspeed_10m_max
  - winddirection_10m_dominant
  - weathercode
  - et0_fao_evapotranspiration
  - date
  - location
  - lat
  - lon


In [7]:
print("5 baris pertama")
print(df_all.head())

print("\n Statistik dasar ")
print(df_all[["temperature_2m_max", "temperature_2m_min", 
              "precipitation_sum", "windspeed_10m_max"]].describe().round(2))

print("\n Missing values ")
print(df_all.isnull().sum())

5 baris pertama
   temperature_2m_max  temperature_2m_min  temperature_2m_mean  \
0                28.3                27.3                 27.8   
1                28.2                27.4                 27.8   
2                28.0                26.3                 27.3   
3                28.3                25.7                 26.8   
4                28.2                27.0                 27.8   

   precipitation_sum  windspeed_10m_max  winddirection_10m_dominant  \
0                0.0                7.4                          79   
1                0.0                8.2                         246   
2                3.3               22.0                         298   
3                7.1               22.8                         293   
4                0.6               24.2                         284   

   weathercode  et0_fao_evapotranspiration       date         location  \
0            3                        4.75 2015-01-01  Balige (Tobasa)   
1           